# FinBERT top-layer fine-tuning (day-level, validation only)

Each day's articles go through FinBERT (top `K` layers trainable, rest frozen), are mean-pooled per article, then per day, concatenated with the 14 market features, and fed to a small head. Day-level loss, so all articles of a day share one label and one split.

- `K = 0` is the frozen-encoder control; `K = 2` is the fine-tuned variant.
- Three expanding validation folds from `fold_roles`. **The test set is never loaded.**
- Epoch is chosen by validation log loss. Last-epoch metrics are reported next to best-epoch metrics to show selection optimism.
- Baselines per fold: training prior and price-only logistic regression.

Jobs (K x seed) run in parallel, one per GPU (threads). Parallel runs are not bit-reproducible; seeds are set but RNG state is shared across threads.

Kaggle: upload `train.csv` and `val.csv` as a dataset, enable Internet (to download FinBERT) and a GPU.

In [ ]:
import os, json, time, warnings, threading, queue
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, matthews_corrcoef, roc_auc_score, log_loss
from transformers import AutoTokenizer, AutoModel
warnings.filterwarnings("ignore")

# ---- config ----
DATA_DIR = Path(os.environ.get("DATA_DIR", "/kaggle/input/usdidr-final"))   # folder with train.csv and val.csv
if not (DATA_DIR / "train.csv").exists():
    DATA_DIR = Path("data/final")
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("ft_output")
OUT_DIR.mkdir(exist_ok=True, parents=True)

ENCODER = os.environ.get("ENCODER", "ProsusAI/finbert")
UNFREEZE = [0, 2]            # number of top transformer layers to train (0 = frozen control)
SEEDS = [42, 123, 2026]
EPOCHS = 4
MAX_LEN, MAX_ART = 128, 16   # tokens per article, articles per day
BATCH_DAYS = 8
LR_ENC, LR_HEAD, WEIGHT_DECAY, DROPOUT = 2e-5, 1e-3, 0.01, 0.3
QUICK_TEST_DAYS = None       # e.g. 80 to smoke-test on a few days per fold

MARKET = ["log_return_t","return_lag1","return_lag2","return_lag3","return_lag5","volatility_5","volatility_20",
          "ma_ratio_5","ma_ratio_20","momentum_5","momentum_10","rsi_14","input_gap_days","decision_weekday"]
device = "cuda" if torch.cuda.is_available() else "cpu"
N_GPU = max(torch.cuda.device_count(), 1)
LOCK = threading.Lock()      # model construction + seeding
GPU_POOL = queue.Queue()
for i in range(N_GPU):
    GPU_POOL.put(f"cuda:{i}" if device == "cuda" else "cpu")
print(device, N_GPU, "GPU(s)", "| data:", DATA_DIR)

In [ ]:
dev = pd.concat([pd.read_csv(DATA_DIR / n, parse_dates=["Date"]) for n in ["train.csv", "val.csv"]])
dev = dev.sort_values("Date").reset_index(drop=True)
dev["articles"] = dev.articles.map(json.loads)
dev["fold_roles"] = dev.fold_roles.map(json.loads)
assert dev.Date.is_unique
print(len(dev), "development days,", sum(map(len, dev.articles)), "articles")

# tokenize each article once
tok = AutoTokenizer.from_pretrained(ENCODER)
texts = {a["article_id"]: a["text_transformer"] for w in dev.articles for a in w}
ids = sorted(texts)
enc_ids = dict(zip(ids, tok([texts[i] for i in ids], truncation=True, max_length=MAX_LEN)["input_ids"]))

# cap articles per day (fixed random subset)
rng0 = np.random.RandomState(0)
day_articles = []
for w in dev.articles:
    a = sorted(x["article_id"] for x in w)
    if len(a) > MAX_ART:
        a = sorted(rng0.choice(a, MAX_ART, replace=False))
    day_articles.append(a)

In [ ]:
class DayModel(nn.Module):
    def __init__(self, k):
        super().__init__()
        self.enc = AutoModel.from_pretrained(ENCODER)
        for p in self.enc.parameters():
            p.requires_grad = False
        self.top = list(self.enc.encoder.layer)[len(self.enc.encoder.layer) - k:] if k else []
        for layer in self.top:
            for p in layer.parameters():
                p.requires_grad = True
        d = self.enc.config.hidden_size
        self.head = nn.Sequential(nn.Linear(d + len(MARKET), 32), nn.ReLU(), nn.Dropout(DROPOUT), nn.Linear(32, 1))

    def set_mode(self, train):
        self.eval()
        if train:
            self.head.train()
            for layer in self.top:
                layer.train()

    def forward(self, input_ids, mask, day_idx, n_days, market):
        h = self.enc(input_ids=input_ids, attention_mask=mask).last_hidden_state
        m = mask.unsqueeze(-1).to(h.dtype)
        art = (h * m).sum(1) / m.sum(1)                                   # article embedding
        art = art.float()
        day = torch.zeros(n_days, art.shape[1], device=art.device).index_add_(0, day_idx, art)
        cnt = torch.bincount(day_idx, minlength=n_days).clamp_min(1).unsqueeze(1).float()
        return self.head(torch.cat([day / cnt, market], 1)).squeeze(-1)


def collate(rows, X, dv):
    seqs, day_idx = [], []
    for j, r in enumerate(rows):
        for a in day_articles[r]:
            seqs.append(enc_ids[a]); day_idx.append(j)
    L = max(len(s) for s in seqs)
    input_ids = torch.zeros(len(seqs), L, dtype=torch.long)
    mask = torch.zeros(len(seqs), L, dtype=torch.long)
    for i, s in enumerate(seqs):
        input_ids[i, :len(s)] = torch.tensor(s); mask[i, :len(s)] = 1
    return (input_ids.to(dv), mask.to(dv), torch.tensor(day_idx, device=dv), len(rows),
            torch.tensor(X[rows], dtype=torch.float32, device=dv))


@torch.no_grad()
def predict(model, rows, X, dv, batch=16):
    model.set_mode(False)
    out = []
    for s in range(0, len(rows), batch):
        with torch.autocast(device, dtype=torch.float16, enabled=device == "cuda"):
            out.append(torch.sigmoid(model(*collate(rows[s:s + batch], X, dv))).float().cpu().numpy())
    return np.concatenate(out)


def score(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return {"log_loss": log_loss(y, p, labels=[0, 1]), "auc": roc_auc_score(y, p),
            "accuracy": accuracy_score(y, p >= 0.5), "mcc": matthews_corrcoef(y, p >= 0.5), "pred_up": float((p >= 0.5).mean())}


def fit(k, seed, tr, va, X, y, dv):
    rng = np.random.RandomState(seed)
    with LOCK:
        torch.manual_seed(seed)
        model = DayModel(k).to(dv)
    enc_params = [p for p in model.enc.parameters() if p.requires_grad]
    groups = [{"params": model.head.parameters(), "lr": LR_HEAD}]
    if enc_params:
        groups.append({"params": enc_params, "lr": LR_ENC})
    opt = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler(enabled=device == "cuda")
    rows = []
    for epoch in range(1, EPOCHS + 1):
        model.set_mode(True)
        order = rng.permutation(tr)
        for s in range(0, len(order), BATCH_DAYS):
            b = order[s:s + BATCH_DAYS]
            with torch.autocast(device, dtype=torch.float16, enabled=device == "cuda"):
                logit = model(*collate(b, X, dv))
            loss = nn.functional.binary_cross_entropy_with_logits(logit.float(), torch.tensor(y[b], dtype=torch.float32, device=dv))
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        rows.append({"epoch": epoch, **score(y[va], predict(model, va, X, dv))})
    del model; torch.cuda.empty_cache()
    return pd.DataFrame(rows)

In [ ]:
y = dev.y.to_numpy(int)
history, baselines = [], []
t0 = time.time()
for fold in ["1", "2", "3"]:
    role = dev.fold_roles.map(lambda r: r[fold])
    tr, va = np.where(role == "train")[0], np.where(role == "val")[0]
    if QUICK_TEST_DAYS:
        tr, va = tr[-QUICK_TEST_DAYS:], va[:QUICK_TEST_DAYS]
    sc = StandardScaler().fit(dev.loc[tr, MARKET])
    X = sc.transform(dev[MARKET]).astype(np.float32)

    prior = np.full(len(va), y[tr].mean())
    lr = LogisticRegression(C=0.1, max_iter=1000).fit(X[tr], y[tr])
    baselines += [{"fold": fold, "method": "train_prior", **score(y[va], prior)},
                  {"fold": fold, "method": "price_only_logreg", **score(y[va], lr.predict_proba(X[va])[:, 1])}]

    def job(k, seed, fold=fold, tr=tr, va=va, X=X):
        dv = GPU_POOL.get()                      # one job per GPU at a time
        try:
            h = fit(k, seed, tr, va, X, y, dv)
        finally:
            GPU_POOL.put(dv)
        h.insert(0, "fold", fold); h.insert(1, "k", k); h.insert(2, "seed", seed)
        b = h.loc[h.log_loss.idxmin()]
        print(f"fold {fold} K={k} seed {seed} [{dv}]: best ep {int(b.epoch)} ll {b.log_loss:.4f} auc {b.auc:.3f} | "
              f"last ll {h.log_loss.iloc[-1]:.4f} | {time.time() - t0:.0f}s", flush=True)
        return h
    with ThreadPoolExecutor(N_GPU) as ex:
        futs = [ex.submit(job, k, seed) for k in UNFREEZE for seed in SEEDS]
        history += [f.result() for f in futs]
    pd.concat(history).to_csv(OUT_DIR / "history.csv", index=False)
pd.DataFrame(baselines).to_csv(OUT_DIR / "baselines.csv", index=False)

In [ ]:
history_df = pd.concat(history)
best = history_df.loc[history_df.groupby(["fold", "k", "seed"]).log_loss.idxmin()]
last = history_df[history_df.epoch == EPOCHS]
cols = ["log_loss", "auc", "accuracy", "mcc", "pred_up"]
summary = pd.concat({
    "best epoch (val log loss)": best.groupby("k")[cols].mean(),
    "last epoch": last.groupby("k")[cols].mean(),
}).round(4)
base = pd.DataFrame(baselines).groupby("method")[cols].mean().round(4)
print("Fine-tuning (mean over folds x seeds)"); display(summary)
print("Baselines (mean over folds)"); display(base)
print("Per fold, best epoch"); display(best.groupby(["fold", "k"])[cols].mean().round(4))
summary.to_csv(OUT_DIR / "summary.csv")

## How to read this

- A variant is only interesting if it beats **both** `train_prior` and `price_only_logreg` on validation log loss in at least 2 of 3 folds, and `K=2` beats `K=0`.
- Compare best-epoch with last-epoch rows: a large gap means validation scores are optimistic from epoch selection.
- Nothing here touches the test set. If a variant clears the bar, freeze its config before any test evaluation.

## Final test evaluation (run once)

Run this **only after** reading the validation results above. Set the frozen config by hand, then run the cells once. Requires `test.csv` in the same dataset.

Procedure: refit on all 930 train+val rows with fixed epochs (no early stopping, no test-based choice), 3 seeds, predict test once, report next to the baselines. Report it honestly in the paper: the test period was already used in earlier experiments (about 15 comparisons), so this is not a clean holdout.

In [ ]:
RUN_TEST = False             # flip to True only after the config below is frozen
FINAL_K = 2                  # chosen from validation (0 or 2)
FINAL_EPOCHS = 2             # chosen from validation: median best epoch across folds/seeds for FINAL_K
print(history_df[history_df.k == FINAL_K].loc[lambda d: d.groupby(["fold","seed"]).log_loss.idxmin()].epoch.describe()[["mean", "50%"]])

In [ ]:
assert RUN_TEST, "Set RUN_TEST = True after freezing FINAL_K / FINAL_EPOCHS"
te = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
te["articles"] = te.articles.map(json.loads)
assert te.Date.min() > dev.Date.max()
n_dev = len(dev)
full = pd.concat([dev, te], ignore_index=True)

# extend tokenization / article caps to test rows
new = {a["article_id"]: a["text_transformer"] for w in te.articles for a in w if a["article_id"] not in enc_ids}
ids2 = sorted(new)
enc_ids.update(dict(zip(ids2, tok([new[i] for i in ids2], truncation=True, max_length=MAX_LEN)["input_ids"])))
for w in te.articles:
    a = sorted(x["article_id"] for x in w)
    if len(a) > MAX_ART:
        a = sorted(rng0.choice(a, MAX_ART, replace=False))
    day_articles.append(a)

yf = full.y.to_numpy(int)
tr_all = np.arange(n_dev)                       # the 930 train+val rows
te_idx = np.arange(n_dev, len(full))
sc = StandardScaler().fit(full.loc[tr_all, MARKET])
Xf = sc.transform(full[MARKET]).astype(np.float32)

def fit_final(k, seed, epochs, dv):
    rng = np.random.RandomState(seed)
    with LOCK:
        torch.manual_seed(seed)
        model = DayModel(k).to(dv)
    enc_params = [p for p in model.enc.parameters() if p.requires_grad]
    groups = [{"params": model.head.parameters(), "lr": LR_HEAD}] + ([{"params": enc_params, "lr": LR_ENC}] if enc_params else [])
    opt = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
    gs = torch.cuda.amp.GradScaler(enabled=device == "cuda")
    for _ in range(epochs):
        model.set_mode(True)
        order = rng.permutation(tr_all)
        for s in range(0, len(order), BATCH_DAYS):
            b = order[s:s + BATCH_DAYS]
            with torch.autocast(device, dtype=torch.float16, enabled=device == "cuda"):
                logit = model(*collate(b, Xf, dv))
            loss = nn.functional.binary_cross_entropy_with_logits(logit.float(), torch.tensor(yf[b], dtype=torch.float32, device=dv))
            opt.zero_grad(); gs.scale(loss).backward(); gs.step(opt); gs.update()
    p = predict(model, te_idx, Xf, dv)               # test touched once, after training ends
    del model; torch.cuda.empty_cache()
    return p

def final_job(s):
    dv = GPU_POOL.get()
    try:
        return fit_final(FINAL_K, s, FINAL_EPOCHS, dv)
    finally:
        GPU_POOL.put(dv)
with ThreadPoolExecutor(N_GPU) as ex:
    probs = dict(zip(SEEDS, ex.map(final_job, SEEDS)))
np.save(OUT_DIR / "test_probs.npy", np.stack(list(probs.values())))

In [ ]:
yt = yf[te_idx]
lr = LogisticRegression(C=0.1, max_iter=1000).fit(Xf[tr_all], yf[tr_all])
streams = {
    f"finbert_K{FINAL_K}": list(probs.values()),
    "train_prior": [np.full(len(yt), yf[tr_all].mean())],
    "price_only_logreg": [lr.predict_proba(Xf[te_idx])[:, 1]],
    "last_direction": [(full.loc[te_idx, "log_return_t"].to_numpy() > 0).astype(float).clip(1e-6, 1 - 1e-6)],
}
rows = [{"method": m, **pd.DataFrame([score(yt, p) for p in ps]).mean().to_dict()} for m, ps in streams.items()]
res = pd.DataFrame(rows).round(4); display(res); res.to_csv(OUT_DIR / "test_results.csv", index=False)

# circular block bootstrap (block 20), conditional on fitted models; seed-averaged probabilities
def boot(p, blocks=20, B=1000, seed=0):
    r = np.random.RandomState(seed); n = len(yt); out = []
    for _ in range(B):
        idx = np.concatenate([(s + np.arange(blocks)) % n for s in r.randint(0, n, int(np.ceil(n / blocks)))])[:n]
        if len(set(yt[idx])) < 2: continue
        out.append((matthews_corrcoef(yt[idx], p[idx] >= .5), roc_auc_score(yt[idx], p[idx])))
    return np.percentile(np.array(out), [2.5, 97.5], axis=0)
pm = np.mean(list(probs.values()), axis=0)
ci = boot(pm)
print(f"ensemble MCC 95% CI [{ci[0,0]:.3f}, {ci[1,0]:.3f}] | AUC 95% CI [{ci[0,1]:.3f}, {ci[1,1]:.3f}]")
print("Majority-class accuracy (always UP):", max(yt.mean(), 1 - yt.mean()).round(4), "| Test up-rate:", yt.mean().round(3), "| model predicted up:", (pm >= .5).mean().round(3))